### EDA of Q1 2025

**Objective**: Find the problem statement & the research questions for the project proposal.

#### 1 - Scope and Configuration

The next cell defines a single frozen configuration object (cfg) that holds every project parameter in one place: the data paths and period, the exposure filters, the prediction horizons H and minimum warning times L, the critical SMART attributes, and the cost ratios. 

It also sets the random seeds for reproducibility, creates the cache and figure folders, and checks that the daily CSV files exist. It also sets an SSD filter and select SMART attributes to focus on.


Later parts of the notebook read from cfg, so moving from Q1 to the full year only requires editing this cell.

In [6]:
from pathlib import Path
from dataclasses import dataclass, field
import random
import numpy as np

@dataclass(frozen=True)
class Config:
    # --- Data location and period ---
    data_dir: Path = Path("../data/data_Q1_2025")      # folder with the daily CSVs (YYYY-MM-DD.csv)
    start_date: str = "2025-01-01"
    end_date: str = "2025-03-31"                    # for the full year: "2025-12-31"
    cache_dir: Path = Path("cache")                 # intermediate results
    fig_dir: Path = Path("figures")

    # --- Scope ---
    hdd_only: bool = True                           # SSD filter
    min_drive_days_per_model: int = 10_000          # exposure filter for AFR reliability (Annualized Failure Rate)
    min_failures_per_model: int = 5                 # minimum failures to report a model-level rate

    # --- Prediction design ---
    horizons_H: tuple = (7, 14, 30)                 # "fails within H days"
    min_warning_L: tuple = (3, 7, 14)               # minimum actionable lead time (days)
    prefailure_window: int = 60                     # days before failure 

    # --- SMART attributes to focus on ---
    critical_smart: tuple = (5, 187, 188, 197, 198)
    other_smart: tuple = (9, 194)                   # power-on hours, temperature

    # --- Cost scenarios (ratio R/M, replacement vs failure cost) ---
    cost_ratios: tuple = (0.05, 0.2, 0.5)

    # --- Reproducibility ---
    seed: int = 42

    # --- Derived helpers ---
    @property
    def critical_cols_raw(self):
        return [f"smart_{i}_raw" for i in self.critical_smart]

    @property
    def critical_cols_norm(self):
        return [f"smart_{i}_normalized" for i in self.critical_smart]

    @property
    def other_cols_raw(self):
        return [f"smart_{i}_raw" for i in self.other_smart]

cfg = Config()

# --- Setup ---
random.seed(cfg.seed)
np.random.seed(cfg.seed)
cfg.cache_dir.mkdir(parents=True, exist_ok=True)
cfg.fig_dir.mkdir(parents=True, exist_ok=True)

# --- Sanity check: are the input files where we expect them? ---
files = sorted(cfg.data_dir.glob("*.csv"))
assert files, f"No CSV files found in {cfg.data_dir.resolve()}"

print(f"Period          : {cfg.start_date} -> {cfg.end_date}")
print(f"Daily files     : {len(files)}  ({files[0].name} ... {files[-1].name})")
print(f"Horizons H      : {cfg.horizons_H}")
print(f"Min warning L   : {cfg.min_warning_L}")
print(f"Critical SMART  : {cfg.critical_smart}")
print(f"HDD only        : {cfg.hdd_only}")

Period          : 2025-01-01 -> 2025-03-31
Daily files     : 90  (2025-01-01.csv ... 2025-03-31.csv)
Horizons H      : (7, 14, 30)
Min warning L   : (3, 7, 14)
Critical SMART  : (5, 187, 188, 197, 198)
HDD only        : True


#### 2 - Loading and Schema audit